In [5]:
# create_sql_query_chain

import os

from langchain_community.utilities import SQLDatabase
from dotenv import load_dotenv
load_dotenv(override=True)
db = SQLDatabase.from_uri(os.getenv("MYSQL_URI")) # type: ignore

print(db.dialect)
print(db.get_usable_table_names())

db.run("SELECT * FROM students LIMIT 5;")

mysql
['courses', 'enrollments', 'students']


"[(1, '张三', '男', 20, 'zhangsan@example.com'), (2, '李四', '女', 21, 'lisi@example.com'), (3, '王五', '男', 19, 'wangwu@example.com')]"

In [6]:
# chain的使用
from langchain_openai import ChatOpenAI

prefix = "DAIPAI_"
model_name = os.getenv(prefix + "MODEL")
model_base_url = os.getenv(prefix + "BASE_URL")
model_api_key = os.getenv(prefix + "API_KEY")

print(f"Model Name: {model_name}")


chat_llm = ChatOpenAI(model=model_name or "gpt-3.5-turbo", base_url=model_base_url, api_key=model_api_key) # type: ignore

Model Name: daipai/qwen/qwen3-next-80b-a3b-thinking


In [7]:
from langchain_classic.chains.sql_database.query import create_sql_query_chain

chain = create_sql_query_chain(chat_llm, db)
responses = chain.invoke({"question": "How many students are in the students table?"})
print(responses)

SQLQuery: SELECT COUNT(*) FROM students LIMIT 5


In [8]:
result = db.run(responses[10:])
print(result)

[(3,)]


In [ ]:
# create_stuff_documents_chain`

from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

# 加载真实的 docx 文件
docx_path = "../1.docx"
loader = UnstructuredWordDocumentLoader(docx_path)
docs = loader.load()

print(f"加载了 {len(docs)} 个文档块")
print(f"文档内容预览：{docs[0].page_content[:100] if docs else '无内容'}")

# 创建提示模板
prompt = ChatPromptTemplate.from_template("""根据以下文档内容，回答问题：

文档：
{context}

问题：{question}

答案：""")

# 使用 create_stuff_documents_chain 创建 chain
stuff_chain = create_stuff_documents_chain(chat_llm, prompt)

# 调用 chain
response = stuff_chain.invoke({"context": docs, "question": "文档中主要讲了什么？"})
print("回答：")
print(response)

加载了 1 个文档块
文档内容预览：3.1.2 知识与能力

IM问题之后的解决方法专注于研究基于内驱力的局限性以及自我平衡理论。这些理论可以分为两个广义理论派别，即基于知识的和基于能力的IM理论如Baldassarre2011；Mi-
回答：


文档主要系统阐述了**内在动机（Intrinsic Motivation, IM）的多学科研究框架**，涵盖理论基础、神经机制、婴幼儿发展过程及机器人/人工智能中的计算模型实现。具体内容包括：

1. **理论基础**：  
   - **基于知识的IM**：分为“新奇性”（通过探索未知扩展知识）和“预测”（通过预测误差驱动学习）两类，强调生物体如何通过环境互动优化认知结构。  
   - **基于能力的IM**：聚焦于技能掌握与自我效能感（如“影响力”“自主性”），关注生物体如何通过挑战性任务提升能力。

2. **神经基础**：  
   - 海马体负责新奇性检测与记忆形成；  
   - 多巴胺通路（腹侧被盖区、上丘等）关联“意外事件”处理与学习信号；  
   - 上丘（SC）通过阶段性多巴胺释放强化行为-结果关联，支撑因果学习。

3. **婴幼儿发展研究**：  
   - **视觉探索**：婴儿从早期“熟悉偏好”转向“新奇偏好”，通过习惯化-去习惯化实验验证；  
   - **预测能力**：通过视觉预期范式（VExP）研究婴儿对动态事件的预测行为（如追踪滚动球）；  
   - **能力发展**：通过“移动实验”等揭示偶然感知、自我效能感的早期发展，以及游戏行为（功能游戏、构造游戏、象征游戏）的阶段性特征。

4. **机器人与AI应用**：  
   - **计算模型**：基于强化学习框架实现IM，如“不确定动机”“信息增益动机”“预测新奇性”“最大化能力进步”等算法；  
   - **实验案例**：  
     - SAIL机器人通过新奇性驱动视觉探索；  
     - AIBO机器人在“游乐场实验”中自主学习技能（如抓取、发声模仿）；  
     - 模拟机器鼠验证上丘-多巴胺机制对行为-结果关联的支撑作用。

文档通过跨学科整合，揭示了IM如何从生物学机制、认知发展到人工系统实现的统一逻辑，强调“内在驱动”对学习与适应的核心作用。
